In [1]:
import itertools as it
import os

import numpy as np

In [2]:
wdir = "/home/b.y.yang/sotiraslab/BradenADLongitudinalPrediction"
odir_base = "/ceph/chpc/shared/aristeidis_sotiras_group/b.y.yang_scratch/BradenADLongitudinalPrediction/revision/new_stables"

# List of parameters

- feature list: combinations of non-imaging, amyloid and volume
- model list: logistic regression, SVM, random forest
- time horizon list: 1yr, 2yr, 3yr, 4yr, 5yr

Notes:
- fixing stable subjects to minimum of 5 years stable
- not allowed to categorize people who progress to MCI/AD past the time window as stable

In [3]:
features_list = ["avn", "av", "an", "vn", "a", "v", "n"]
# features_list = ["a", "v", "n"]
# features_list = ["avn"]
model_list = ["svm"]
time_window_list = [1,2,3,4,5]
leave_out_group = ["site", "tracer"]

# Create commands

In [4]:
cmd_list = []

### without ComBat

In [5]:
# odir = os.path.join(odir_base, "base")
odir = odir_base

for f, m, t, g in it.product(features_list, model_list, time_window_list, leave_out_group):

    prefix = f"{f}__{m}__{t}yr__{g}"

    cmd = f"python train.py -w '{wdir}' -o '{odir}' --prefix '{prefix}' --leave_out_group {g} -f {f} -m {m} -t {t} -s 5 --random_state 42 --control_standardize"
    
    cmd_list.append(cmd)

### with ComBat tracer harmonization (site LOGO only)

In [6]:
odir = os.path.join(odir_base, "combat")

for f, m, t, g in it.product(features_list, model_list, time_window_list, leave_out_group):

    prefix = f"{f}__{m}__{t}yr__{g}"

    if g != "site": continue

    cmd = f"python train.py -w '{wdir}' -o '{odir}' --prefix '{prefix}' --leave_out_group {g} -f {f} -m {m} -t {t} -s 5 --random_state 42 --control_standardize --combat"
    
    cmd_list.append(cmd)

### without ComBat, ensemble model

In [7]:
odir = os.path.join(odir_base, "ensemble")

for f, m, t, g in it.product(features_list, model_list, time_window_list, leave_out_group):

    prefix = f"{f}__{m}__{t}yr__{g}"

    cmd = f"python train.py -w '{wdir}' -o '{odir}' --prefix '{prefix}' --leave_out_group {g} -f {f} -m {m} -t {t} -s 5 --random_state 42 --control_standardize --ensemble --n_estimators 100"
    
    cmd_list.append(cmd)

### with ComBat tracer harmonization, ensemble model (site LOGO only)

In [8]:
odir = os.path.join(odir_base, "combat_ensemble")

for f, m, t, g in it.product(features_list, model_list, time_window_list, leave_out_group):

    prefix = f"{f}__{m}__{t}yr__{g}"

    if g != "site": continue

    cmd = f"python train.py -w '{wdir}' -o '{odir}' --prefix '{prefix}' --leave_out_group {g} -f {f} -m {m} -t {t} -s 5 --random_state 42 --control_standardize --ensemble --n_estimators 100 --combat"
    
    cmd_list.append(cmd)

### For revisions: use the same time window definition for stables and progressors

In [6]:
# odir = "/ceph/chpc/shared/aristeidis_sotiras_group/b.y.yang_scratch/BradenADLongitudinalPrediction/revision/stable_time_window"
odir = odir_base

for f, m, t, g in it.product(features_list, model_list, time_window_list, leave_out_group):

    prefix = f"{f}__{m}__{t}yr__{g}"

    cmd = f"python train.py -w '{wdir}' -o '{odir}' --prefix '{prefix}' --leave_out_group {g} -f {f} -m {m} -t {t} -s {t} --random_state 42 --control_standardize --alt_stable"
    
    cmd_list.append(cmd)

### For revisions: train single-tracer models

In [13]:
odir = "/ceph/chpc/shared/aristeidis_sotiras_group/b.y.yang_scratch/BradenADLongitudinalPrediction/revision/single_tracer"

for f, m, t, g, tracer in it.product(["avn"], ["svm"], [1,2,3,4,5], ["site"], ["FBP", "PIB"]):

    prefix = f"{f}__{m}__{t}yr__{g}"

    cmd = f"python train.py -w '{wdir}' -o '{odir}' --prefix '{prefix}' --leave_out_group {g} -f {f} -m {m} -t {t} -s 5 --random_state 42 --control_standardize --single_tracer {tracer}"
    
    cmd_list.append(cmd)

# Save file

In [8]:
np.savetxt(
    "/home/b.y.yang/sotiraslab/BradenADLongitudinalPrediction/code/3_TrainTestModel/slurm/cmd_new_stables.txt",
    np.array(cmd_list),
    fmt = "%s"
)